
Analytical Platform Knowledge Base - Query Interface & Comparison Demo

This notebook provides an interactive interface for querying the Analytical Platform
documentation using AWS Bedrock Knowledge Base with optional metadata filtering.

================================================================================
#### PRIMARY QUERY FUNCTIONS (from utils.py)
================================================================================

Use these functions to interact with the Knowledge Base:

1. ask() - User-friendly interface with formatted output
   • Best for: Interactive exploration in notebooks
   • Shows: Formatted answer, clickable sources, metadata
   • Example:
     ask("How do I use RStudio?")
     ask("How do I delete a table?", metadata_filters=MetadataFilters.by_heading("Athena"))

2. chat() - Programmatic interface returning raw data
   • Best for: Automation, scripts, batch processing
   • Returns: (answer_text, citations_list) tuple
   • Example:
     answer, citations = chat("How do I install packages?")
     process_answer(answer)

3. retrieve_and_generate_with_sources() - Low-level API access
   • Best for: Advanced debugging, custom workflows
   • Returns: Full API response with metadata
   • Example:
     response = retrieve_and_generate_with_sources(
         query="test",
         metadata_filters={...},
         verbose=True
     )

================================================================================
#### COMPARISON DEMO FUNCTIONS (this notebook)
================================================================================

Compare query results with and without metadata filters to understand their impact:

MAIN FUNCTION:
    demo_comparison(question, metadata_filters, show_mode)
    
    Executes the SAME query twice:
    1. WITH your specified metadata filters
    2. WITHOUT any filters (baseline)
    
    Then displays side-by-side comparison to show filtering impact.

DISPLAY MODES:
    - "compact" (default): Answers + comparison stats in clean vertical layout
    - "detailed": Adds full source citations for each result
    - "side-by-side": Table-based comparison with enhanced metrics

USAGE EXAMPLES:

    # Basic comparison - see impact of filtering
    demo_comparison(
        "How do I install packages?",
        metadata_filters={"root_heading": {"equals": "RStudio"}}
    )
    
    # Detailed mode - includes full sources
    demo_comparison(
        "How do I delete a table?",
        metadata_filters=MetadataFilters.by_heading("Athena"),
        show_mode="detailed"
    )
    
    # Side-by-side mode - tabular comparison
    demo_comparison(
        "What are the data access policies?",
        metadata_filters={"page_url": {"stringContains": "aup.html"}},
        show_mode="side-by-side"
    )

WHAT THE COMPARISON SHOWS:
    ✓ How filters affect number of sources retrieved
    ✓ Impact on answer length and detail
    ✓ Whether filters narrow results appropriately
    ✓ If filters are too restrictive (excluding relevant info)
    ✓ Difference in answer focus and specificity

TYPICAL WORKFLOW IN THIS NOTEBOOK:

    Step 1: Ask a question without filters (baseline)
    -------
    ask("How do I query data?")
    
    Step 2: If results are too broad, add metadata filters
    -------
    ask("How do I query data?", 
        metadata_filters=MetadataFilters.by_heading("Athena"))
    
    Step 3: Compare filtered vs unfiltered results
    -------
    demo_comparison(
        "How do I query data?",
        metadata_filters=MetadataFilters.by_heading("Athena"),
        show_mode="detailed"
    )
    
    Step 4: Refine filters based on comparison insights
    -------
    # Too restrictive? Relax filters
    # Still too broad? Add more specific filters
    # Just right? Use in production queries

USE CASES FOR COMPARISON DEMOS:
    - Testing metadata filter effectiveness
    - Demonstrating retrieval precision to stakeholders
    - Debugging over-filtering or under-filtering issues
    - Comparing answer quality with different filter strategies
    - POC evaluation: showing value of metadata filtering
    - Training users on optimal filter usage

RETURNS:
    Dictionary with both results for programmatic access:
    {
        'filtered': (answer, citations),
        'unfiltered': (answer, citations)
    }
    
    This allows further analysis:
    results = demo_comparison("question", filters={...})
    filtered_answer = results['filtered'][0]
    unfiltered_citations = results['unfiltered'][1]

INTERNAL FUNCTIONS:
    _render_results()      : Routes to appropriate display format
    _render_standard()     : Vertical layout (compact/detailed modes)
    _render_side_by_side() : Table-based comparison
    _display_sources()     : Format and display citation sources

================================================================================
#### POC NOTE
================================================================================

The demo_comparison() functions are designed for:
    • POC evaluation and stakeholder demonstrations
    • Understanding metadata filter behavior
    • Debugging retrieval quality issues

For production chatbot/Q&A systems, users would typically:
    • Use ask() or chat() directly (single query, not comparison)
    • Apply filters based on predetermined rules or user context
    • Show only one result (filtered or unfiltered, not both)

The comparison mode helps you DECIDE which filtering strategy to use in production,
but is not typically part of the end-user experience.

================================================================================
#### GETTING STARTED
================================================================================

1. Run the system check cell below to verify connectivity
2. Try basic queries with ask()
3. Experiment with metadata filters using MetadataFilters helper
4. Use demo_comparison() to understand filter impact
5. Iterate on filter strategies based on comparison insights

See sections below for examples and interactive demos.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from typing import Dict, List, Optional, Tuple
from IPython.display import display, Markdown
load_dotenv()

# Add project root to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from helpers.apug.old_rag.vector_search_05 import ask, chat, MetadataFilters, display_citations_table, export_qa_to_markdown
from helpers.apug.old_rag.query_evaluator_05 import demo_comparison, _render_results,_render_standard,_render_side_by_side, _display_sources


#### question-1

In [ ]:
question1 ="On the topic of the data uploader. Is there a process to request the removal of an existing table? " \
    "I have 2 tables that got data ingested and are now in Athena. But those tables changed during the latest designs review and now they are obsolete. " \
    "How can I get those deleted? What is the process to request such action? " \
    "Worth to highlight that the data stored in those tables is replaceable and there is no need to keep an archive of it"

demo_comparison(question1, metadata_filters={'page_h1' : {'equals': "Data Uploader - Analytical Platform User Guidance"}}, show_mode="detailed")

#### **Question1 summary:**

"On the topic of the data uploader. Is there a process to request the removal of an existing table? " I have 2 tables that got data ingested and are now in Athena. But those tables changed during the latest designs review and now they are obsolete. How can I get those deleted? What is the process to request such action? Worth to highlight that the data stored in those tables is replaceable and there is no need to keep an archive of it.

#### **Corresponding link:**
https://user-guidance.analytical-platform.service.justice.gov.uk/tools/data-uploader/index.html

#### **Reason for not answering:**

The Data Uploader guidance explains how to upload and manage tables but does not mention any process for removing or deleting tables once they are in Athena.
It also doesn’t state whether deletion is supported or if you need to raise a request through support.



#

#

#### question-2

In [ ]:
question2="Hi team, I am looking to create a new S3 bucket for a new data pipeline I am building. " \
"Do we have a dos/ don'ts list somewhere on which options to select or ok to just follow the default AWS options recommends." \
"E.g.:Object ownership should be ACLS disabled or acls enabled"

demo_comparison(question2, metadata_filters={'page_h1' : {'equals': "Amazon S3 - Analytical Platform User Guidance"}}, show_mode= 'side-by-side')


#### **Question2 summary:**

Hi team, I am looking to create a new S3 bucket for a new data pipeline I am building. Do we have a dos/ don'ts list somewhere on which options to select or ok to just follow the default AWS options recommends. E.g.:Object ownership should be ACLS disabled or acls enabled"

#### **Corresponding link:**
https://user-guidance.analytical-platform.service.justice.gov.uk/tools/data-uploader/index.html

##### **Reason for not answering:**

While this page explains how to access data in S3 and use tools like r, s3browser and botor, it does not include:

- Instructions for creating new S3 buckets
- A dos/don’ts list for bucket configuration
- Guidance on object ownership settings (ACLs enabled/disabled) or other security best practices

#### question-3

In [ ]:
question3="Hello, Iam looking for some airflow help! We have a DAG in the DE Prod airflow which has been failing the last couple of days" \
      "but when I try to make a release of the fix using the existing DE github workflow the github action just sits in a queue for ages so the image" \
         " is not building. I thought the turn off was not until next month, has something else changed where we can not make new releases to the old airflow?"

demo_comparison(question3, metadata_filters={'page_h1' : {'equals': "Airflow Concepts - Analytical Platform User Guidance"}}, show_mode='compact')


#### **question 3 summary:**

"Hello, Iam looking for some airflow help! We have a DAG in the DE Prod airflow which has been failing the last couple of days but when I try to make a release of the fix using the existing DE github workflow the github action just sits in a queue for ages so the image is not building. I thought the turn off was not until next month, has something else changed where we can not make new releases to the old airflow?"

#### **Corresponding link:**

https://user-guidance.analytical-platform.service.justice.gov.uk/tools/airflow/concepts/index.html

**Reason for not answering:**

While the guidance documents that Data Engineering Airflow is deprecated and scheduled for shutdown, it does not explicitly state:

 - That new releases are no longer possible before the shutdown date
 - That GitHub workflows will stop functioning or queue indefinitely prior to the official turn-off

#

#

#### question-4

In [ ]:
question4="Hello! I've run my airflow pipeline and it's failed as the container seems to be killed due to a spacing issue." \
    " I've had a look here and it reaches a peak of about 11.6GiB and the CPU hit about 2.02, I changed it to general-spot-2vcpu-16gb in a PR" \
         "(though I think I will up the CPU to 4 if possible as well) but the github action failed saying there's an unallowed value there,"

demo_comparison(question4, metadata_filters={'page_h1' : {'equals': "Airflow Concepts - Analytical Platform User Guidance"}})

#### question-4

"Hello! I've run my airflow pipeline and it's failed as the container seems to be killed due to a spacing issue. I've had a look here and it reaches a peak of about 11.6GiB and the CPU hit about 2.02, I changed it to general-spot-2vcpu-16gb in a PR" \(though I think I will up the CPU to 4 if possible as well) but the github action failed saying there's an unallowed value there,"

#### What the guidance includes:

- Overview of Airflow concepts on the Analytical Platform
- Instructions for creating workflows, DAGs, and using BasicKubernetesPodOperator
- General deployment steps and GitHub release process
- Notes on Airflow migration and deprecation

#### What the guidance does not include

- A list of allowed/valid resource classes (e.g., which CPU/memory presets are permitted for AP Airflow workflows). No mapping of names like general-spot-2vcpu-16gb is documented.
- Guidance on how to increase CPU/memory or valid presets for AP Airflow
- Steps to troubleshoot GitHub Action errors specifically about “unallowed value” for resource configuration in AP Airflow.
- Instructions on how to request higher CPU/memory or which labels to use to avoid container OOM/CPU kill events.

#### question-5

In [ ]:
question5 = "I’m reaching out on behalf of a colleague who is currently locked out of Slack." \
    "Could you advise on how to publish or share an RShiny app in a way that allows it to be continuously maintained in the background, " \
    "while also providing a no-code interface for others to interact with?Any guidance or recommended platforms would be greatly appreciated."
demo_comparison(question5, metadata_filters={'page_h1' : {'equals': "Apps- r-shiny-apps"}}) 
# Deploy an RShiny app - Analytical Platform User Guidance correct page_h1

#### question-5 summary

"Hello! I've run my airflow pipeline and it's failed as the container seems to be killed due to a spacing issue. I've had a look here and it reaches a peak of about 11.6GiB and the CPU hit about 2.02, I changed it to general-spot-2vcpu-16gb in a PR" \(though I think I will up the CPU to 4 if possible as well) but the github action failed saying there's an unallowed value there,"

#### What the guidance includes:

- Overview of Airflow concepts on the Analytical Platform
- Instructions for creating workflows, DAGs, and using BasicKubernetesPodOperator
- General deployment steps and GitHub release process
- Notes on Airflow migration and deprecation

#### What the guidance does not include

- A list of allowed/valid resource classes (e.g., which CPU/memory presets are permitted for AP Airflow workflows). No mapping of names like general-spot-2vcpu-16gb is documented.
- Guidance on how to increase CPU/memory or valid presets for AP Airflow
- Steps to troubleshoot GitHub Action errors specifically about “unallowed value” for resource configuration in AP Airflow.
- Instructions on how to request higher CPU/memory or which labels to use to avoid container OOM/CPU kill events.

#### question-6

In [ ]:
question6= "Quick question about our repos." \
                        "I've noticed most projects use main as the default branch while others use the legacy master nomenclature." \
                        "is ther a recommended standard for what we should be using as the head/default branch?" \
                        "Re- lgacy repos. should we continue sticking with master, or should we migrating them to main?"

demo_comparison(question6, metadata_filters={'page_h1' : {'equals': "Tools - Analytical Platform User Guidance"}})                     

#### question-6

"Quick question about our repos.I've noticed most projects use main as the default branch while others use the legacy master nomenclature. is ther a recommended standard for what we should be using as the head/default branch? Re- lgacy repos. should we continue sticking with master, or should we migrating them to main?"

#### What the guidance includes:

- Instructions for using GitHub within the Analytical Platform
- How to create repositories, manage workflows, and make releases
- General best practices for working with GitHub Actions

#### What the guidance does not include

- Any explicit recommendation for default branch naming conventions (e.g., main vs master)
- Guidance on whether legacy repos should migrate from master to main
- A formal standard for head/default branch naming across projects

#### question-7

In [ ]:
question7="Hi, we've had a couple of task failures in our pipelines on AP-airflow Production, " \
    "they seem to be getting stuck in a 'pod has failed to load' loop until they hit a 600 second time-out. " \
    "It appears to only be affecting tasks where we're asking for elevated compute (we've experienced it with both spot and on-demand). " \
    "Is this a known issue? Currently it looks like we've just got to keep restarting the task until it behaves."

demo_comparison(question7, metadata_filters={'page_h1' : {'equals': "Airflow Concepts - Analytical Platform User Guidance"}})     

#### question-8

In [ ]:
question8 = "We have deployed services using the analytical platform for our private beta MVP release, for which we registered the application in" \
    "analytical platform.Now we are moving away from analytical platform and using our own helm to deploy the application." \
    "we created a new namespace for testing purpose, but since the application is already registered in analytical platform, " \
    " we could see the namespace details over there. So if we need to move away from analytical platform and deploy our existing" \
    "applications with our own helm charts, what is the procedure to follow?"

demo_comparison(question8, metadata_filters={'page_h1' : {'equals': "Airflow Concepts - Analytical Platform User Guidance"}})   

#### question-9

In [ ]:
question9 = "Good morning, all - I am unable to log into R Studio today (v4.5.1-1). I keep getting an error message that says" \
    "502 Bad Gateway" \
    "or sometimes" \
    "504 Gateway Time-out." \
        "Nothing i try is resolving this problem: restarting the laptop; refreshing the webpage; clearing the browser cache." \
            " Some helpful advice would be much appreciated!"
demo_comparison(question9, metadata_filters={'page_h1' : {'equals': "Airflow Concepts - Analytical Platform User Guidance"}})  

#### question-10

In [ ]:
question10 = " QuickSight question: I have changed the schema of my data in Athena. This doesn't automatically propagate to QuickSight." \
    " Is there a way of updating the schema of QS datasets, without having to delete and readd datasets? We do not use SPICE yet." \
    "My understanding is that a view is taken of the data, so perhaps the data source configuration can be kicked for our views to reload the schema?"
demo_comparison(question10, metadata_filters={'page_h1' : {'equals': "Airflow Concepts - Analytical Platform User Guidance"}})  

#### **Conclusion:**

The current Knowledge Base (KB) provides basic and core operational guidance for standard workflows, which works well for routine tasks. However, colleagues frequently encounter edge cases, decision-making scenarios, and troubleshooting issues that are not documented. This gap forces them to rely on Slack discussions for answers, creating inefficiencies and inconsistent knowledge sharing.

#### **Recommendation**

To address this challenge:

1. Integrate Slack knowledge into the KB
 - Capture questions and solutions from Slack that are not covered in the KB.
 - Convert resolved Slack threads into KB articles using a simple, consistent template.

2. Expand KB coverage beyond basic instructions
 - Include decision-making guidance, troubleshooting steps, and frequently asked questions.
 - Ensure KB evolves to reflect real-world scenarios colleagues face.

3. Establish a continuous improvement process
 - Monitor Slack for recurring questions and log them as KB gaps.
 - Regularly review and update KB content to close these gaps.

4. Enhance chatbot logic
 - Route queries to KB first.
 - If KB lacks the answer, provide a clear fallback message and record the gap for future KB enrichment.

##### **Outcome:**

By integrating Slack knowledge and expanding KB coverage, colleagues will have a single, authoritative source for both routine and complex queries, reducing dependency on ad-hoc Slack support and improving efficiency.

#### Technical recommendations

##### 1) **Session Management (Multi‑turn Context)**

Enables the chatbot to remember prior questions, applied filters, and user preferences—unlocking coherent, multi-question interactions.  

**What to implement:**

*   A session store (e.g., Redis/Postgres) keyed by `session_id`
*   Persisted context (filters, role) and recent conversation history
*   A context composer that selectively includes relevant turns in each answer

##### 2) **Validation Helper (KB-only Guardrail)**

Ensures every answer is grounded **only** in your KB—prevents off-KB content and maintains trust.  
**What to implement:**

*   Post-generation checks that all claims have KB citations
*   Block external domains and disallowed phrasing
*   Auto-fallback to your standardized “cannot answer” template if validation fails

##### 3) **Score Threshold Filtering (Confidence Gate)**

Stops low-confidence answers before they reach users; improves perceived quality.  
**What to implement:**

*   Aggregate score from vector similarity, keyword match, and recency weighting
*   Tunable `MIN_SCORE` per intent/topic
*   If below threshold → return KB-only fallback and log a KB gap

##### 4) **Enhanced Metadata Filtering (Boolean Combinations)**

Increases retrieval precision for complex queries (e.g., combining tags/sections/owners).  
**What to implement:**

*   Metadata schema for chunks (tags, section, owner, updated\_at)
*   Support `AND` / `OR` / `NOT` filters in the retrieval pipeline
*   Re-rank filtered candidates before answer generation

##### 5) **Response Caching (Speed & Consistency)**

**Why fifth:** Cuts latency and cost for repeated or popular queries; provides stable outputs.  
**What to implement:**

*   Cache key built from `(normalized_query, filters, role, KB_version)`
*   TTL + invalidation on KB updates (hook on content merge)
*   Store both retrieval results and final rendered answer


